# Centaur IME: 韓国語の模型の学習（GPU、1 言語だけ）

言語ごとに別のタブで同時に動かすためのノート。全部の言語を 1 冊で順に回すのは train_all.ipynb。

使い方: 「ランタイム → ランタイムのタイプを変更」で **L4 GPU**、「ランタイム → すべてのセルを実行」。
**最初のセルで Google ドライブにつなぐ**（許可する。飛ばさない）。結果はドライブ（CentaurIME/models/2026-11-07）に書き、
最後にランタイムを手放す。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/CentaurIME/models/2026-11-07'
import os; os.makedirs(OUT, exist_ok=True); print('保存先', OUT)

In [ ]:
!rm -rf repo && git clone -q --depth 1 -b work/lang-measure https://github.com/J-KAMMA/centaur-ime.git repo && git -C repo log -1 --format='%H %s'
!pip -q install pyarrow==21.0.0

In [ ]:
!python repo/tools/fetch_source.py tatoeba-kor data/ko
!python repo/tools/fetch_source.py wiki-ko data/ko
!python repo/tools/fetch_source.py cc100-ko data/ko --lines 300000

In [ ]:
!python repo/tools/models/train_task.py --task ko --tatoeba data/ko/kor_sentences.tsv.bz2 --wiki data/ko/train-00002-of-00003.parquet --cc100 data/ko/cc100-ko.300000.txt --extra 300000 --d 384 --layers 8 --heads 6 --batch 256 --minutes 45 --amp --out "$OUT"
!ls -la "$OUT" | grep ko
print(open(OUT + '/ko-result.txt', encoding='utf-8').read())

In [ ]:
from google.colab import runtime
runtime.unassign()